In [ ]:
# CELL 1 — ONLY USER CONTROL
RUN_SEED = 1337  # CHANGE ONLY THIS: 42 / 1337 / 2026 / 31415 / 271828

PUBLICATION_SEEDS = [42, 1337, 2026, 31415, 271828]
if RUN_SEED not in PUBLICATION_SEEDS:
    raise ValueError(f"RUN_SEED must be one of {PUBLICATION_SEEDS}, got {RUN_SEED}")

MODEL_ID = "google/paligemma2-10b-pt-448"
MODEL_REVISION = "cf10265ef3ed7a410e3abe989e74e44d0c803176"

# [Verified] validation-selected old Gated checkpoints from the completed five-seed PaliGemma2 runs.
GATED_BEST_STEP_BY_SEED = {
    42: 800,
    1337: 800,
    2026: 932,
    31415: 932,
    271828: 932,
}

SHARED_FORWARD_PROTOCOL_SHA256 = "37b7d30a994fb98225d49d76cca5768a838d456abf844dc1a5d93019a0cd15aa"
FACETGATE_COMMON_SHA256 = "735b3e08413949cdb148fe4ce570493b5cd410e562dad7a12748b2e5f2fd6a38"
PARENT_GATED_PROTOCOL_SHA256 = "627bdf5a77ab694581ff590c32f8885c901ebad09e8df86dbab07762caaf95dd"

print({
    "RUN_SEED": RUN_SEED,
    "expected_old_gated_best_step": GATED_BEST_STEP_BY_SEED[RUN_SEED],
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "TEST_forwards": 0,
})


In [ ]:
# CELL 2 — CPU-ONLY SAME-SEED OLD-GATED DISCOVERY
# This cell does not load PaliGemma2, does not use GPU, and does not touch TEST.

import json, hashlib, zipfile
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

EXPECTED_STEP = int(GATED_BEST_STEP_BY_SEED[RUN_SEED])
EXPECTED_CP_NAMES = {
    f"checkpoint-{EXPECTED_STEP}",
    f"checkpoint-{EXPECTED_STEP:04d}",
}

def _sha256_file(path, chunk=8*1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def _find_root_from_checkpoint(cp):
    cp = Path(cp)
    if cp.parent.name == "checkpoints":
        return cp.parent.parent
    return cp.parent

def _load_json(path):
    try:
        return json.loads(Path(path).read_text(encoding="utf-8"))
    except Exception:
        return None

def _run_config_ok(obj):
    if not isinstance(obj, dict):
        return False, "run_config_missing_or_invalid"
    if int(obj.get("seed", -1)) != int(RUN_SEED):
        return False, f"run_config_seed={obj.get('seed')}"
    if str(obj.get("model_id", "")) != MODEL_ID:
        return False, f"run_config_model_id={obj.get('model_id')}"
    if str(obj.get("model_revision", "")) != MODEL_REVISION:
        return False, f"run_config_model_revision={obj.get('model_revision')}"
    if obj.get("frozen_common_gated_protocol_sha256") != PARENT_GATED_PROTOCOL_SHA256:
        return False, "wrong_parent_gated_protocol_sha256"
    # The frozen training recipe itself must remain development-only even if a
    # later separate locked-test controller consumed the frozen checkpoint.
    if obj.get("allow_final_test") not in (False, None):
        return False, f"run_config_allow_final_test={obj.get('allow_final_test')}"
    return True, "ok"

def _checkpoint_manifest_ok(obj):
    if not isinstance(obj, dict):
        return False, "checkpoint_manifest_missing_or_invalid"
    if int(obj.get("seed", -1)) != int(RUN_SEED):
        return False, f"checkpoint_seed={obj.get('seed')}"
    if int(obj.get("global_step", -1)) != EXPECTED_STEP:
        return False, f"checkpoint_step={obj.get('global_step')}"
    if obj.get("frozen_common_gated_protocol_sha256") != PARENT_GATED_PROTOCOL_SHA256:
        return False, "checkpoint_protocol_sha_mismatch"
    fp = obj.get("checkpoint_content_sha256")
    if not (isinstance(fp, str) and len(fp) == 64):
        return False, "checkpoint_content_sha256_missing"
    return True, "ok"

def _threshold_payload_ok(obj):
    if not isinstance(obj, dict):
        return False, "threshold_json_invalid"
    if obj.get("source_split") not in (None, "validation"):
        return False, f"threshold_source_split={obj.get('source_split')}"
    if obj.get("source_rows") not in (None, 932):
        return False, f"threshold_source_rows={obj.get('source_rows')}"
    th = obj.get("thresholds", obj)
    if not isinstance(th, dict):
        return False, "threshold_mapping_missing"
    return True, "ok"

def _directory_candidate(cp):
    cp = Path(cp)
    required = [
        cp / "classifier_heads.pt",
        cp / "adapter" / "adapter_config.json",
        cp / "checkpoint_manifest.json",
    ]
    adapter_ok = (
        (cp / "adapter" / "adapter_model.safetensors").is_file()
        or (cp / "adapter" / "adapter_model.bin").is_file()
    )
    if not all(x.is_file() for x in required) or not adapter_ok:
        return None, "checkpoint_files_incomplete"

    root = _find_root_from_checkpoint(cp)
    rc_path = root / "run_config.json"
    rc = _load_json(rc_path)
    ok, reason = _run_config_ok(rc)
    if not ok:
        return None, reason

    mf = _load_json(cp / "checkpoint_manifest.json")
    ok, reason = _checkpoint_manifest_ok(mf)
    if not ok:
        return None, reason

    threshold_candidates = [
        root / "artifacts" / "thresholds.json",
        root / "thresholds.json",
        root / "artifacts" / "validation_thresholds.json",
        root / "validation_thresholds.json",
    ]
    th_path = next((p for p in threshold_candidates if p.is_file()), None)
    if th_path is None:
        return None, "validation_thresholds_missing"
    th_obj = _load_json(th_path)
    ok, reason = _threshold_payload_ok(th_obj)
    if not ok:
        return None, reason

    return {
        "kind": "directory",
        "source": str(cp),
        "root": str(root),
        "thresholds": str(th_path),
        "run_config": str(rc_path),
        "manifest": str(cp / "checkpoint_manifest.json"),
        "fingerprint": str(mf["checkpoint_content_sha256"]),
    }, "ok"

def _zip_candidates(zp):
    rows = []
    try:
        with zipfile.ZipFile(zp) as zf:
            names = set(zf.namelist())
            manifests = [
                n for n in names
                if n.endswith("checkpoint_manifest.json")
                and n.rstrip("/").split("/")[-2] in EXPECTED_CP_NAMES
            ]
            for mm in manifests:
                cp_prefix = mm[:-len("checkpoint_manifest.json")]
                root_prefix = cp_prefix.split("checkpoints/")[0]

                required = {
                    cp_prefix + "classifier_heads.pt",
                    cp_prefix + "adapter/adapter_config.json",
                }
                adapter_ok = (
                    cp_prefix + "adapter/adapter_model.safetensors" in names
                    or cp_prefix + "adapter/adapter_model.bin" in names
                )
                if not required.issubset(names) or not adapter_ok:
                    continue

                rc_member = root_prefix + "run_config.json"
                if rc_member not in names:
                    # Resume safeguards normally store run_config.json at ZIP root.
                    rc_member = "run_config.json" if "run_config.json" in names else None
                if rc_member is None:
                    continue
                rc = json.loads(zf.read(rc_member).decode("utf-8"))
                ok, reason = _run_config_ok(rc)
                if not ok:
                    continue

                mf = json.loads(zf.read(mm).decode("utf-8"))
                ok, reason = _checkpoint_manifest_ok(mf)
                if not ok:
                    continue

                threshold_members = [
                    root_prefix + "artifacts/thresholds.json",
                    root_prefix + "thresholds.json",
                    "artifacts/thresholds.json",
                    "thresholds.json",
                ]
                th_member = next((n for n in threshold_members if n in names), None)
                if th_member is None:
                    continue
                th_obj = json.loads(zf.read(th_member).decode("utf-8"))
                ok, reason = _threshold_payload_ok(th_obj)
                if not ok:
                    continue

                # Extract only the threshold JSON for downstream explicit binding.
                th_out = WORK_ROOT / f"_paligemma_old_gated_seed{RUN_SEED}_{mf['checkpoint_content_sha256'][:12]}_thresholds.json"
                th_out.write_bytes(zf.read(th_member))

                rows.append({
                    "kind": "zip",
                    "source": str(zp),
                    "checkpoint_prefix": cp_prefix,
                    "thresholds": str(th_out),
                    "run_config_member": rc_member,
                    "manifest_member": mm,
                    "fingerprint": str(mf["checkpoint_content_sha256"]),
                })
    except Exception:
        pass
    return rows

discovered = []
rejected = []

# Direct notebook-output directories.
if INPUT_ROOT.exists():
    for mf in INPUT_ROOT.rglob("checkpoint_manifest.json"):
        cp = mf.parent
        if cp.name not in EXPECTED_CP_NAMES:
            continue
        row, reason = _directory_candidate(cp)
        if row is not None:
            discovered.append(row)
        else:
            rejected.append({"source": str(cp), "reason": reason})

# Portable Resume_Safeguard / paper ZIP mirrors.
if INPUT_ROOT.exists():
    for zp in INPUT_ROOT.rglob("*.zip"):
        for row in _zip_candidates(zp):
            discovered.append(row)

# Collapse directory + ZIP copies of the SAME checkpoint by the frozen
# checkpoint_content_sha256. Prefer a directly mounted directory.
groups = {}
for r in discovered:
    groups.setdefault(r["fingerprint"], []).append(r)

parents = []
mirrors = []
for fp, group in groups.items():
    group = sorted(
        group,
        key=lambda r: (
            0 if r.get("kind") == "directory" else 1,
            len(str(r.get("source", ""))),
            str(r.get("source", "")),
        ),
    )
    chosen = dict(group[0])
    chosen["mirror_count"] = len(group)
    chosen["mirror_sources"] = [
        (x.get("source"), x.get("checkpoint_prefix"))
        for x in group
    ]
    parents.append(chosen)
    if len(group) > 1:
        mirrors.append({
            "checkpoint_content_sha256": fp,
            "canonical_source": chosen["source"],
            "mirror_count": len(group),
        })

if len(parents) > 1:
    raise RuntimeError(
        "More than one DIFFERENT valid same-seed PaliGemma2 Gated checkpoint "
        "was found. Directory/ZIP mirrors with the same content hash are allowed, "
        "but different checkpoint hashes are not.\n"
        + json.dumps(parents, indent=2)
    )

PARENT = parents[0] if parents else None
EXECUTION_MODE = (
    "REUSE_EXISTING_GATED"
    if PARENT is not None
    else "SIMULTANEOUS_FROM_SCRATCH"
)

controller_record = {
    "schema": "M3Offend_9318_PaliGemma2_10B_PT448_FaceGate_AUTO_SEED_v1_0_controller",
    "seed": int(RUN_SEED),
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "expected_old_gated_best_step": EXPECTED_STEP,
    "execution_mode": EXECUTION_MODE,
    "parent": PARENT,
    "duplicate_mirrors_collapsed": mirrors,
    "rejected_candidates": rejected[:100],
    "facetgate_common_sha256": FACETGATE_COMMON_SHA256,
    "shared_forward_protocol_sha256": SHARED_FORWARD_PROTOCOL_SHA256,
    "parent_gated_protocol_sha256": PARENT_GATED_PROTOCOL_SHA256,
    "TEST_forwards": 0,
}
controller_path = WORK_ROOT / f"paligemma_facegate_controller_seed{RUN_SEED}.json"
controller_path.write_text(json.dumps(controller_record, indent=2), encoding="utf-8")

print("[CONTROLLER DECISION]", json.dumps(controller_record, indent=2))
if PARENT:
    print(
        f"[OLD GATED FOUND] Seed={RUN_SEED} step={EXPECTED_STEP} "
        f"fingerprint={PARENT['fingerprint']} mirrors={PARENT.get('mirror_count',1)}"
    )
    print("[MODE] REUSE_EXISTING_GATED -> Gated frozen -> FaceGate/SHTE Target only")
else:
    print("[OLD GATED NOT FOUND]")
    print("[MODE] SIMULTANEOUS_FROM_SCRATCH -> one Pali forward/micro-batch -> Gated + detached FaceGate Target")


In [ ]:
# CELL 3 — EMBEDDED REVIEWED EXECUTION PATHS
# The outer controller chooses exactly one child after the CPU-only precheck.

SIMULTANEOUS_CHILD_ZLIB_B64 = '...'
TARGET_ONLY_CHILD_ZLIB_B64 = '...'

SIMULTANEOUS_CHILD_SHA256 = '7453c632c61109dcf44671e339d2105a22257f3bfac50cdc43f1a46473b9f35e'
TARGET_ONLY_CHILD_SHA256 = 'daa47c7e1a539d937240ac9542147fcfe46eeebf2ed15bea48807b1310768f5d'

print({
    "simultaneous_child_sha256": SIMULTANEOUS_CHILD_SHA256,
    "target_only_child_sha256": TARGET_ONLY_CHILD_SHA256,
    "TEST_forwards": 0,
})


In [ ]:
# CELL 4 — MATERIALIZE THE SELECTED SEED-SPECIFIC CHILD
import base64, zlib, json, hashlib, re
from pathlib import Path

def _decode_child(blob, expected_sha):
    raw = zlib.decompress(base64.b64decode(blob.encode("ascii")))
    observed = hashlib.sha256(raw).hexdigest()
    if observed != expected_sha:
        raise RuntimeError(
            f"Embedded child SHA mismatch expected={expected_sha} observed={observed}"
        )
    return json.loads(raw.decode("utf-8"))

if EXECUTION_MODE == "REUSE_EXISTING_GATED":
    child = _decode_child(TARGET_ONLY_CHILD_ZLIB_B64, TARGET_ONLY_CHILD_SHA256)
else:
    child = _decode_child(SIMULTANEOUS_CHILD_ZLIB_B64, SIMULTANEOUS_CHILD_SHA256)

for cell in child.get("cells", []):
    if cell.get("cell_type") != "code":
        continue

    src = "".join(cell.get("source", []))
    src = src.replace("__RUN_SEED__", str(int(RUN_SEED)))

    if EXECUTION_MODE == "REUSE_EXISTING_GATED":
        # Inject only into the child CONFIG cell.  Later cells legitimately
        # reassign GATED_SOURCE_OVERRIDE/GATED_THRESHOLDS_OVERRIDE at runtime,
        # so patching every occurrence would corrupt the orchestrator.
        if re.search(
            r'(?m)^\s*AUTO_RUN_GATED_IF_MISSING\s*=',
            src,
        ):
            src, n_src = re.subn(
                r'(?m)^\s*GATED_SOURCE_OVERRIDE\s*=.*$',
                "GATED_SOURCE_OVERRIDE=" + repr(str(PARENT["source"])),
                src,
                count=1,
            )
            src, n_thr = re.subn(
                r'(?m)^\s*GATED_THRESHOLDS_OVERRIDE\s*=.*$',
                "GATED_THRESHOLDS_OVERRIDE=" + repr(str(PARENT["thresholds"])),
                src,
                count=1,
            )
            src, n_auto = re.subn(
                r'(?m)^\s*AUTO_RUN_GATED_IF_MISSING\s*=.*$',
                "AUTO_RUN_GATED_IF_MISSING=False",
                src,
                count=1,
            )
            if n_src != 1 or n_thr != 1 or n_auto != 1:
                raise RuntimeError(
                    "Failed to inject controller-bound old-Gated parent into "
                    f"target-only config cell: source={n_src}, thresholds={n_thr}, auto={n_auto}"
                )

    cell["source"] = src.splitlines(keepends=True)
    cell["execution_count"] = None
    cell["outputs"] = []

name = (
    f"M3Offend_9318_PaliGemma2_10B_PT448_FaceGate_Seed{RUN_SEED}_TARGET_ONLY.ipynb"
    if EXECUTION_MODE == "REUSE_EXISTING_GATED"
    else f"M3Offend_9318_PaliGemma2_10B_PT448_FaceGate_Seed{RUN_SEED}_FULL.ipynb"
)

# Semantic preflight of the generated child BEFORE any subprocess/GPU work.
_generated_source = "\n".join(
    "".join(c.get("source", []))
    for c in child.get("cells", [])
    if c.get("cell_type") == "code"
)
if "__RUN_SEED__" in _generated_source:
    raise RuntimeError("Generated child still contains unresolved __RUN_SEED__ placeholder.")

if EXECUTION_MODE == "REUSE_EXISTING_GATED":
    if repr(str(PARENT["source"])) not in _generated_source:
        raise RuntimeError("Generated target-only child is missing bound Gated source path.")
    if repr(str(PARENT["thresholds"])) not in _generated_source:
        raise RuntimeError("Generated target-only child is missing bound Gated threshold path.")
    if "CONTROLLER-BOUND REUSE MODE" not in _generated_source:
        raise RuntimeError("Generated target-only child is not the controller-bound v2 child.")
    if re.search(r'(?m)^\s*AUTO_RUN_GATED_IF_MISSING\s*=\s*True\s*$', _generated_source):
        raise RuntimeError("Target-only child unexpectedly enables Gated fallback.")
else:
    required_sim_tokens = [
        "SIMULTANEOUS_FROM_SCRATCH",
        "class FacetGateTarget",
        "Shared-Forward SHTE update",
        "fixed_gated_validation_cache",
    ]
    missing = [x for x in required_sim_tokens if x not in _generated_source]
    if missing:
        raise RuntimeError(f"Generated simultaneous child missing invariants: {missing}")

CHILD_NOTEBOOK = WORK_ROOT / name
CHILD_NOTEBOOK.write_text(
    json.dumps(child, ensure_ascii=False, indent=1),
    encoding="utf-8",
)

# Fail before expensive execution if any generated code cell has a syntax error.
for i, cell in enumerate(child["cells"]):
    if cell.get("cell_type") == "code":
        compile(
            "".join(cell.get("source", [])),
            f"{name}:cell{i}",
            "exec",
        )

CHILD_SHA256 = hashlib.sha256(CHILD_NOTEBOOK.read_bytes()).hexdigest()

print("[CHILD STATIC QA PASS]", {
    "seed": RUN_SEED,
    "execution_mode": EXECUTION_MODE,
    "child_notebook": str(CHILD_NOTEBOOK),
    "child_sha256": CHILD_SHA256,
    "TEST_forwards": 0,
})


In [ ]:
# CELL 5 — EXECUTE THE AUTOMATICALLY SELECTED PIPELINE
# The child gets a fresh Python kernel so PaliGemma2 package/runtime changes
# cannot contaminate this lightweight controller kernel.

import subprocess, sys, time
from pathlib import Path

EXECUTED_NOTEBOOK = CHILD_NOTEBOOK.with_name(
    CHILD_NOTEBOOK.stem + "_EXECUTED.ipynb"
)

cmd = [
    sys.executable,
    "-m",
    "jupyter",
    "nbconvert",
    "--to",
    "notebook",
    "--execute",
    "--ExecutePreprocessor.timeout=-1",
    "--output",
    EXECUTED_NOTEBOOK.name,
    str(CHILD_NOTEBOOK),
]

print("[EXECUTING]", " ".join(cmd), flush=True)
started = time.time()

proc = subprocess.run(cmd, cwd=str(WORK_ROOT))

elapsed = time.time() - started

if proc.returncode != 0:
    raise RuntimeError(
        f"Selected PaliGemma2 FaceGate pipeline failed: "
        f"mode={EXECUTION_MODE} returncode={proc.returncode}. "
        f"Inspect {EXECUTED_NOTEBOOK if EXECUTED_NOTEBOOK.exists() else CHILD_NOTEBOOK}"
    )

print("[PIPELINE COMPLETE]", {
    "seed": RUN_SEED,
    "execution_mode": EXECUTION_MODE,
    "elapsed_seconds": elapsed,
    "executed_notebook": str(EXECUTED_NOTEBOOK),
    "TEST_forwards": 0,
})


In [ ]:
# CELL 6 — FINAL CONTROLLER SUMMARY
import json
from pathlib import Path

print("=" * 100)
print("M3Offend PaliGemma2 FaceGate AUTO-SEED COMPLETE")
print("=" * 100)
print("Seed :", RUN_SEED)
print("Mode :", EXECUTION_MODE)
print("Executed notebook :", EXECUTED_NOTEBOOK)

if EXECUTION_MODE == "REUSE_EXISTING_GATED":
    print("Scientific route : old same-seed Gated reused/frozen; exact FacetGate v1.0 Target-only stage")
    print("Parent checkpoint :", PARENT["source"])
    print("Parent fingerprint :", PARENT["fingerprint"])
else:
    root = Path(
        f"/kaggle/working/M3Offend_9318_PaliGemma2_10B_PT448_FaceGate_Seed{RUN_SEED}"
    )
    manifest = root / "artifacts" / "dual_checkpoint_final_manifest.json"
    if manifest.is_file():
        result = json.loads(manifest.read_text(encoding="utf-8"))
        print("Scientific route : full shared-forward Gated + FaceGate/SHTE from scratch")
        print("Selected Gated step :", result.get("selected_gated_step"))
        print("Selected FaceGate step :", result.get("selected_facetgate_step"))
        print("Validation metrics :", json.dumps(result.get("validation_metrics"), indent=2))
    else:
        print("WARNING: expected final shared-forward manifest not found:", manifest)

print("Locked TEST forwards : 0")
print("=" * 100)
